## 3. MACHINE LEARNING FOR CLASSIFICATION

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

pd.set_option('display.max_columns', None)

df = pd.read_csv('telco_customer_churn.csv', sep=',')

print(f'columns: {df.shape[1]}\nrows: {df.shape[0]}')
display(df.head(2).T)
display(df.describe().T)
##display(df.dtypes)

columns: 21
rows: 7043


,0,1
customerID,7590-VHVEG,5575-GNVDE
gender,Female,Male
SeniorCitizen,0,0
Partner,Yes,No
Dependents,No,No
tenure,1,34
PhoneService,No,Yes
MultipleLines,No phone service,No
InternetService,DSL,DSL
OnlineSecurity,No,Yes


,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


### 3.2 DATA PREPARATION

In [26]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
string_list = list(df.dtypes[df.dtypes == 'str'].index)

for col in string_list:
    df[col] = df[col].str.lower().str.replace(' ', '_').str.replace('-', '_')

# column example with string value that should be numeric
# let fix it by replacing with 0 and then convert to numeric
display(df[df.index == 488].T)

df['totalcharges'] = pd.to_numeric(df['totalcharges'].replace('_', '0'), errors='coerce')
df['monthlycharges'] = pd.to_numeric(df['monthlycharges'].replace('_', '0'), errors='coerce')
df['seniorcitizen'] = df['seniorcitizen'].astype(int).replace(0, 'no').replace(1, 'yes').astype(str)

#df['CHURN'] = df['CHURN'].map({'yes': 1, 'no': 0})
df['churn'] = (df['churn'] == 'yes').astype(int)

,488
customerid,4472_lvygi
gender,female
seniorcitizen,0
partner,yes
dependents,yes
tenure,0
phoneservice,no
multiplelines,no_phone_service
internetservice,dsl
onlinesecurity,yes


### 3.3 SETTING THE VALIDATION FRAMEWORK

In [27]:
# split the data into 80% train, 20% test, 20% of the train will be used as validation
# 20% of 80% is 16% of the total data
# so we use 20%/80% = 25% of the train data as validation
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['churn']
y_val = df_val['churn']
y_test = df_test['churn']

X_train = df_train.drop(columns=['churn'])
X_val = df_val.drop(columns=['churn'])
X_test = df_test.drop(columns=['churn'])

print(f'train: {len(df_train)}\nvalidation: {len(df_val)}\ntest: {len(df_test)}')
display((y_train.value_counts(normalize=True)*100).round(2))

train: 4225
validation: 1409
test: 1409


churn
0    73.14
1    26.86
Name: proportion, dtype: float64

In [28]:
categorical_columns = X_train.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
categorical_columns.remove('customerid')
numerical_columns = X_train.select_dtypes(exclude=['object', 'category', 'str']).columns.tolist()

display(X_train[numerical_columns].head(3).T)
display(X_train[categorical_columns].head(3).T)

,0,1,2
tenure,72.00,10.00,5.00
monthlycharges,115.50,95.25,75.55
totalcharges,8425.15,1021.55,413.65


,0,1,2
gender,female,male,female
seniorcitizen,no,no,no
partner,yes,no,no
dependents,yes,no,no
phoneservice,yes,yes,yes
multiplelines,yes,yes,yes
internetservice,fiber_optic,fiber_optic,fiber_optic
onlinesecurity,yes,no,no
onlinebackup,yes,yes,no
deviceprotection,yes,yes,no


### 3.5 FEATURE IMPORTANCE: CHURN RATE AND RISK RATIO

#### 3.5.1 CHURN RATE

In [29]:
## GLOBAL CHURN
global_churn = df_full_train['churn'].mean()

## GENDER CHURN
churn_male = df_full_train[df_full_train['gender'] == 'male']['churn'].mean()
churn_female = df_full_train[df_full_train['gender'] == 'female']['churn'].mean()
print(f'female churn: {churn_female:.2%}\nmale churn: {churn_male:.2%}\nglobal churn: {global_churn:.2%}\n')

## PARTNER CHURN
churn_partner_yes = df_full_train[df_full_train['partner'] == 'yes']['churn'].mean()
churn_partner_no = df_full_train[df_full_train['partner'] == 'no']['churn'].mean()
print(f'partner yes churn: {churn_partner_yes:.2%}\npartner no churn: {churn_partner_no:.2%}\npartner global churn: {global_churn:.2%}\n')

female churn: 27.68%
male churn: 26.32%
global churn: 27.00%

partner yes churn: 20.50%
partner no churn: 32.98%
partner global churn: 27.00%



The following snippet displays the value of the global **churn rate**. In comparison to that value, we can also calculate the churn rates for the **female** and **male** groups. We observe that the female churn rate is slightly higher than the global rate, while the male churn rate is slightly lower than the global rate.

When examining **partner** group, we notice that customers with partners are significantly less likely to churn. The churn rate for this group is **approximately 20%**, contrasting with the global churn rate of **almost 27%**. On the other hand, customers without partners have a much higher churn rate compared to the global rate, **standing at 33%** as opposed to 27%.

This observation suggests that the **partner variable may be more influential** for predicting churn **than the gender variable**.

### 3.5.2 RISK RATIO

In the context of machine learning and classification, the “risk ratio” typically refers to a statistical measure used to assess the likelihood or probability of a certain event occurring in one group compared to another. It’s a useful concept in various fields, including healthcare, finance, and customer churn analysis.

In the specific context of churn rate, the risk ratio can help you understand the relative risk of churn (i.e., customers leaving) for different groups or segments within your dataset. It can provide insights into which features or factors are associated with a higher or lower risk of churn.

Here’s a simplified explanation of how risk ratio works in the context of churn rate:

>**1. Definition of Risk Ratio**: The risk ratio (also known as the relative risk) is defined as the probability of an event occurring in one group divided by the probability of the same event occurring in another group. In the case of churn rate, you’re typically comparing two groups: one group that exhibits a certain characteristic or behavior (e.g., customer has churned) and another group that does not exhibit that characteristic (e.g., customer hasn’t churned).

>**2. Interpretation**: A risk ratio greater than 1 suggests that the event (churn in this case) is more likely in the first group compared to the second group. A risk ratio less than 1 suggests the event is less likely in the first group. A risk ratio equal to 1 means there is no difference in risk between the two groups.

>**3. Application**: We can use risk ratios to assess the impact of different features or interventions on churn rate. For example, we might calculate the risk ratio of churn for customers who received a promotional offer versus those who did not. If the risk ratio is significantly greater than 1, it indicates that the promotional offer had a positive impact on reducing churn.

>**4. Statistical Significance**: It’s important to also consider statistical significance when interpreting risk ratios. Statistical tests such as chi-squared tests or confidence intervals can help determine if the observed differences in churn rates are statistically significant.

So the risk ratio is a valuable tool for assessing the impact of different factors or features on churn rate in classification tasks. It helps you quantify and compare the relative risk of churn between different groups, providing insights that can inform decision-making and strategies for reducing churn.

Let’s compare the risk ratio for churning between people with partners and those without partners.

In [30]:
print(f'partner no churn ratio: {churn_partner_no / global_churn:.2%}')
print(f'partner yes churn ratio: {churn_partner_yes / global_churn:.2%}')

partner no churn ratio: 122.17%
partner yes churn ratio: 75.95%


This demonstrates that the churn rate for **people without partners is 22% higher, whereas for people with partners, it is 24% lower than the global churn rate**.

Let’s take the data and group it by gender, and for each variable within the gender group, let’s calculate the average churn rate within that group and calculate the difference and risk. We can perform this analysis for all the variables, not just the gender variable.

In [31]:
df_full_train.groupby('partner')['churn'].mean()
df_full_train.groupby('partner')['churn'].agg(['mean', 'count'])

,mean,count
partner,,
no,0.329809,2932
yes,0.205033,2702


In [32]:
for col in categorical_columns:
    df_group = df_full_train.groupby(col)['churn'].agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn
    df_group['risk'] = df_group['mean'] / global_churn
    display(df_group)

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


,mean,count,diff,risk
seniorcitizen,,,,
no,0.242270,4722,-0.027698,0.897403
yes,0.413377,912,0.143409,1.531208


,mean,count,diff,risk
partner,,,,
no,0.329809,2932,0.059841,1.221659
yes,0.205033,2702,-0.064935,0.759472


,mean,count,diff,risk
dependents,,,,
no,0.313760,3968,0.043792,1.162212
yes,0.165666,1666,-0.104302,0.613651


,mean,count,diff,risk
phoneservice,,,,
no,0.241316,547,-0.028652,0.893870
yes,0.273049,5087,0.003081,1.011412


...

,mean,count,diff,risk
streamingtv,,,,
no,0.342832,2246,0.072864,1.269897
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.302723,2167,0.032755,1.121328


,mean,count,diff,risk
streamingmovies,,,,
no,0.338906,2213,0.068938,1.255358
no_internet_service,0.077805,1221,-0.192163,0.288201
yes,0.307273,2200,0.037305,1.138182


,mean,count,diff,risk
contract,,,,
month_to_month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


Concerning the difference, we calculate it as the group’s churn rate minus the global churn rate **_(Δ churn = group - global)_**. This metric represents the **absolute deviation in percentage points (p.p.)** from the overall average.

- **Positive values (> 0)** indicate that the group has a higher churn rate than the global average (worse performance).

- **Negative values (< 0)** indicate that the group has a lower churn rate than the global average (better performance).

The magnitude of the value reflects **how far the group is from the average in absolute terms**.

This metric is particularly useful for identifying **which groups contribute most to overall churn deterioration**, as it captures real impact in percentage points rather than proportional differences.

_It is important to note that this measure is descriptive, not causal: it indicates whether a group is above or below the average, but does not, by itself, explain why the difference exists._

For a more complete analysis, this metric is often used alongside the relative ratio (group / global), which captures proportional deviation. While the difference highlights absolute impact, the ratio highlights relative intensity, and both should be interpreted together for better decision-making.

---

As for the **risk ratio**, it is obtained by **dividing the group’s churn rate by the global churn rate _(risk ratio = group / global)_**. This metric represents the **relative deviation from the overall average**.

- **Values greater than 1 (> 1)** indicate that the group has a higher churn rate than the global average (worse performance).

- **Values less than 1 (< 1)** indicate that the group has a lower churn rate than the global average (better performance).

- A value of **exactly 1 indicates that the group is aligned** with the global average.

The **risk ratio captures how many times higher (or lower) the churn rate is relative to the baseline**. For example, a value of 1.5 means the group’s churn rate is 50% higher than average, while 0.8 indicates it is 20% lower than average.

_It is important to emphasize that, similarly to the difference metric, the risk ratio is descriptive rather than causal. It reflects how a group compares to the average, but does not explain the underlying drivers of churn._

---

In essence, both difference (group − global) and risk ratio (group / global) convey related information from complementary perspectives:

>`The difference measures absolute impact (in percentage points)`

>`The risk ratio measures relative intensity (proportional change)`

Using both together provides a more complete understanding of how each group behaves compared to the overall population.

By analyzing these metrics across categories, **we can identify which groups systematically deviate from the global churn behavior** - either positively or negatively. These group-level patterns highlight **potentially relevant features for churn analysis and modeling**.

However, evaluating categories in isolation is not sufficient to determine the overall importance of a variable. A variable may show strong deviations in some categories but still have limited impact at the population level (for example, if those categories are small).

To compare variables such as “contract” and “streamingmovies” in terms of their overall relevance, we need a more aggregated measure of importance. This requires combining the magnitude of deviation with the distribution (size) of each group, which we will address in the next steps.

### 3.6 FEATURE IMPORTANCE: MUTUAL INFORMATION (CATEGORICAL)

In [33]:
def mutual_information_score(series):
    return mutual_info_score(series, df_full_train.churn)

mutual_information = df_full_train[categorical_columns].apply(mutual_information_score).sort_values(ascending=False)
mutual_information

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

### 3.7 FEATURE IMPORTANCE: CORRELATION (NUMERICAL)

In [59]:
display(df_full_train[numerical_columns].corrwith(df_full_train['churn']).sort_values(ascending=True))
display(df_full_train[numerical_columns].corrwith(df_full_train['churn']).abs().sort_values(ascending=False))

tenure           -0.351885
totalcharges     -0.196353
monthlycharges    0.196805
dtype: float64

tenure            0.351885
monthlycharges    0.196805
totalcharges      0.196353
dtype: float64

In [44]:
display(df_full_train[numerical_columns].max())

tenure              72.00
monthlycharges     118.65
totalcharges      8684.80
dtype: float64

In [ ]:
monthly_charges_ranges = [[0, 20],
                          [20, 50],
                          [50, 70],
                          [70, 90],
                          [90, 120],
                          [50, 120]] # 50 to 120 because the max is 118.75

tenure_ranges = [[0, 2],
                 [2, 12],
                 [12, 24],
                 [24, 48],
                 [48, 72],
                 [12, 72]] # 12 to 72 because the max is 72

for start, end in tenure_ranges:
    churn_rate = df_full_train[(df_full_train['tenure'] > start) &
                               (df_full_train['tenure'] <= end)]['churn'].mean()
    print(f"tenure range [{start}, {end}] months: {churn_rate * 100:.4f}%")

print()

for start, end in monthly_charges_ranges:
    churn_rate = df_full_train[(df_full_train['monthlycharges'] > start) &
                               (df_full_train['monthlycharges'] <= end)]['churn'].mean()
    print(f"monthly charges range [{start}, {end}] dollars: {churn_rate * 100:.4f}%")

tenure range [0, 2] months: 60.2356%
tenure range [2, 12] months: 39.9441%
tenure range [12, 24] months: 30.1724%
tenure range [24, 48] months: 20.4651%
tenure range [48, 72] months: 9.8250%
tenure range [12, 72] months: 17.6349%

monthly charges range [0, 20] dollars: 8.7954%
monthly charges range [20, 50] dollars: 18.3409%
monthly charges range [50, 70] dollars: 21.8143%
monthly charges range [70, 90] dollars: 38.5135%
monthly charges range [90, 120] dollars: 33.2135%
monthly charges range [50, 120] dollars: 32.4993%
